# CNN for MNIST

This notebook trains a compact convolutional neural network on the MNIST digits dataset. The goal is to understand how local image patterns are learned through filters, pooling, and dense layers.

## Understanding the Problem

MNIST is a classic image classification dataset containing handwritten digits from 0 to 9. Each image is small, grayscale, and easy to inspect visually, which makes it an ideal dataset for learning the basics of CNNs.

## Core Concept

A CNN learns spatial features such as edges, corners, and loops by applying filters across the image. After feature extraction, the network flattens the learned representation and classifies it into one of 10 digit classes.

## Mathematical Intuition

A convolution layer computes:

$$
Y = X * K + b
$$

where $X$ is the input image and $K$ is the convolution kernel. ReLU introduces non-linearity, while pooling reduces dimensionality and keeps the most important activations.

## Model Architecture

The model used here follows this structure:

Input
↓
Conv2D
↓
ReLU
↓
MaxPooling
↓
Flatten
↓
Dense
↓
Softmax


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Add channel dimension: (samples, height, width, channels)
x_train = x_train[..., None]
x_test = x_test[..., None]

# One-hot encode labels
num_classes = 10
y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)

# Show a few sample digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.show()

# Build CNN
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Train model
history = model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

# Plot learning curves
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.title('Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history.history['accuracy'], label='Training accuracy')
plt.plot(history.history['val_accuracy'], label='Validation accuracy')
plt.title('Accuracy Curves')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Evaluate on test set
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f'Test loss: {loss:.4f}')
print(f'Test accuracy: {accuracy:.4f}')

# Confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_pred_prob = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Digit')
plt.ylabel('Actual Digit')
plt.show()

# Sample predictions
sample_indices = np.random.choice(len(x_test), 9, replace=False)
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, idx in zip(axes.flat, sample_indices):
    image = x_test[idx].squeeze()
    prediction = y_pred[idx]
    actual = y_true[idx]
    ax.imshow(image, cmap='gray')
    ax.set_title(f'Actual {actual}\nPred {prediction}', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

# Technical analysis
print('The CNN learns local image structure using convolution filters and compresses the representation with pooling before classification.')
print('The final dense layer outputs a probability distribution across the 10 digit classes.')
